## The unified read/write API

Spark reads and writes data through a single fluent API:

```python
# Read
df = spark.read.format("parquet").option("...", "...").load("path/")

# Write
df.write.format("parquet").mode("overwrite").save("path/")
```

The format-shorthand methods — `.csv()`, `.json()`, `.parquet()`, `.orc()` — are aliases for the longer form. Same plumbing, less typing.

Five formats matter for everyday Spark work:

| Format | Schema source | Best for |
|---|---|---|
| **CSV** | Header row or explicit | Human-readable, slow, text |
| **JSON** | Inferred or explicit | Semi-structured (loan-origination payloads, API dumps) |
| **Parquet** | Embedded in file footer | Columnar, default for analytics |
| **ORC** | Embedded in file footer | Columnar, common in Hadoop pipelines |
| **Delta** | Embedded + transaction log | Lakehouse storage with ACID, time travel |

![Spark Data Sources](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-data-sources.svg)

## Setup

The `data/` folder is generated by `generate_bank_data.ipynb` (see the apache-spark CLAUDE.md). Eight tables across five formats — CSV, JSON, Parquet, ORC, Delta — match the canonical schemas in `project/schemas/bank_schemas.py`.

All write examples in this notebook land in `data/_write_examples/` so the source data stays untouched.

In [ ]:
import os
import shutil
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

DATA_DIR  = "data"
OUT_DIR   = os.path.join(DATA_DIR, "_write_examples")
WAREHOUSE = os.path.abspath(os.path.join(DATA_DIR, "_warehouse"))
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(WAREHOUSE, exist_ok=True)

builder = (
    SparkSession.builder
    .appName("ReadingWritingData")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.warehouse.dir", WAREHOUSE)   # used by saveAsTable in §11
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("data/ contents:", sorted(os.listdir(DATA_DIR)))

## Reading the five formats

One inline cell per format, each reading a real bank table from `data/`.

In [ ]:
# CSV — customers.
# `header=true` reads the first row as column names.
# Always pass an explicit schema in production — inference is slow and can guess wrong.
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, TimestampType,
    BooleanType, DecimalType, IntegerType,
)

customers_schema = StructType([
    StructField("customer_id",   StringType(), nullable=False),
    StructField("full_name",     StringType()),
    StructField("email",         StringType()),
    StructField("phone",         StringType()),
    StructField("date_of_birth", DateType()),
    StructField("gender",        StringType()),
    StructField("city",          StringType()),
    StructField("state",         StringType()),
    StructField("country",       StringType()),
    StructField("created_at",    TimestampType()),
])

customers = (
    spark.read
    .schema(customers_schema)
    .option("header", "true")
    .option("sep",    ",")        # default; explicit for clarity
    .csv(f"{DATA_DIR}/customers")
)

customers.show(3, truncate=False)
print("rows:", customers.count())

In [ ]:
# JSON — card_accounts.
# Each line is one JSON object (line-delimited JSON, the default).
# Use option("multiLine", "true") if each record spans multiple lines.
card_accounts_schema = StructType([
    StructField("card_id",      StringType(),       nullable=False),
    StructField("customer_id",  StringType(),       nullable=False),
    StructField("card_type",    StringType()),
    StructField("credit_limit", DecimalType(18, 2)),
    StructField("status",       StringType()),
    StructField("issued_at",    TimestampType()),
])

card_accounts = (
    spark.read
    .schema(card_accounts_schema)
    .json(f"{DATA_DIR}/card_accounts")
)

card_accounts.show(3, truncate=False)

In [ ]:
# Parquet — card_transactions.
# Parquet embeds its schema in the file footer — Spark reads it automatically.
# Passing an explicit schema is still useful: enforce types, or select a column subset
# (predicate / projection pushdown then reads less data off disk).
card_transactions = spark.read.parquet(f"{DATA_DIR}/card_transactions")
card_transactions.printSchema()
card_transactions.show(3, truncate=False)
print("rows:", card_transactions.count())

In [ ]:
# ORC — bank_accounts.
# ORC (Optimized Row Columnar) is columnar like Parquet, common in legacy Hadoop pipelines.
# Same pattern: schema is embedded; pass an explicit one to enforce types if you wish.
bank_accounts = spark.read.orc(f"{DATA_DIR}/bank_accounts")
bank_accounts.printSchema()
bank_accounts.show(3, truncate=False)

In [ ]:
# Delta — payments.
# A Delta table is a directory of parquet files plus a _delta_log/ folder.
# Pass the path to the directory; the transaction log handles the rest.
payments = spark.read.format("delta").load(f"{DATA_DIR}/payments")
payments.printSchema()
payments.show(3, truncate=False)

## Schema inference vs explicit

| | Explicit schema | Inferred schema |
|---|---|---|
| Speed | Fast — no scan | Slow — full or sampled scan |
| Type safety | Guaranteed | Best-effort guess |
| Null handling | Declared per column | Assumed nullable |
| Recommended for | Production pipelines | Exploratory work only |

Inference triggers a *separate scan over the data* before your query starts running. On a 100 GB CSV that's a non-trivial pre-job. Worse: a single malformed row can change the inferred type silently — `credit_limit` was `decimal` yesterday; today one row contains the string `"NA"` and it's now `string`, and downstream `sum()` calls fail.

Always pass an explicit schema in production.

In [ ]:
import time

# Inferred — Spark scans the whole file to guess types
t0 = time.perf_counter()
inferred = (
    spark.read
    .option("header",       "true")
    .option("inferSchema",  "true")
    .csv(f"{DATA_DIR}/customers")
)
inferred.count()
print(f"inferred : {time.perf_counter() - t0:.2f}s; first 3 dtypes = {inferred.dtypes[:3]}")

# Explicit — no scan, just read with the schema we declared above
t0 = time.perf_counter()
explicit = (
    spark.read
    .schema(customers_schema)
    .option("header", "true")
    .csv(f"{DATA_DIR}/customers")
)
explicit.count()
print(f"explicit : {time.perf_counter() - t0:.2f}s; first 3 dtypes = {explicit.dtypes[:3]}")

## Bad-record handling — the `mode` option

When a row in your input doesn't conform to the schema, Spark needs a policy.

| `mode` | Behavior |
|---|---|
| `PERMISSIVE` (default) | Set the bad row's failed columns to `null`; keep the row. With `columnNameOfCorruptRecord` set, the raw bad string is stored in that column. |
| `DROPMALFORMED` | Silently drop the bad row. |
| `FAILFAST` | Throw an exception on the first bad row. |

Set `badRecordsPath` to write rejected rows to a separate location for offline inspection — useful in production.

In [ ]:
# Build a tiny CSV with one good row and one malformed row.
sample_csv = os.path.join(OUT_DIR, "_sample.csv")
with open(sample_csv, "w") as f:
    f.write("id,amount\n")
    f.write("1,100\n")
    f.write("2,not-a-number\n")
    f.write("3,300\n")

mini_schema = StructType([
    StructField("id",     IntegerType(), nullable=False),
    StructField("amount", IntegerType(), nullable=True),
])

# PERMISSIVE — bad row's amount becomes null
print("--- PERMISSIVE ---")
(
    spark.read.schema(mini_schema)
    .option("header", "true").option("mode", "PERMISSIVE")
    .csv(sample_csv).show()
)

# DROPMALFORMED — bad row dropped
print("--- DROPMALFORMED ---")
(
    spark.read.schema(mini_schema)
    .option("header", "true").option("mode", "DROPMALFORMED")
    .csv(sample_csv).show()
)

# FAILFAST — would raise. Catch the exception so the cell still completes.
print("--- FAILFAST ---")
try:
    (
        spark.read.schema(mini_schema)
        .option("header", "true").option("mode", "FAILFAST")
        .csv(sample_csv).count()
    )
except Exception as e:
    print(f"raised: {type(e).__name__}")

## Reading multiple files — globs and directories

A path that points at a directory reads every file under it (which is what every cell so far has done — `data/customers/` had multiple `part-*.csv` files inside). For finer control:

- **Glob patterns** — `*.csv`, `customers/part-*.parquet`. Standard `*` and `?` wildcards work.
- **Multiple paths** — `.load(["a/", "b/"])` reads both as one DataFrame, as long as schemas match.
- **`recursiveFileLookup=true`** — descend into subdirectories instead of treating them as Hive partitions.

In [ ]:
# Build a small directory tree with a shared schema so the demo isn't dependent on layout.
glob_dir = os.path.join(OUT_DIR, "glob_demo")
shutil.rmtree(glob_dir, ignore_errors=True)
os.makedirs(glob_dir, exist_ok=True)
for name, body in [("a.csv", "id,name\n1,alice\n2,bob\n"),
                   ("b.csv", "id,name\n3,carol\n4,dan\n")]:
    with open(os.path.join(glob_dir, name), "w") as f:
        f.write(body)

# Nest one more file inside a subfolder for the recursive demo.
nested_dir = os.path.join(glob_dir, "nested", "deep")
os.makedirs(nested_dir, exist_ok=True)
with open(os.path.join(nested_dir, "c.csv"), "w") as f:
    f.write("id,name\n5,eve\n6,frank\n")

# Glob — read both top-level CSVs at once
g1 = spark.read.option("header", "true").csv(f"{glob_dir}/*.csv")
print("glob       :", g1.count(), "rows")

# Multiple explicit paths — same effect via list-of-paths
g2 = spark.read.option("header", "true").csv([
    os.path.join(glob_dir, "a.csv"),
    os.path.join(glob_dir, "b.csv"),
])
print("list-of-paths:", g2.count(), "rows")

# recursiveFileLookup — pull every CSV under glob_dir, including the nested one
g3 = (
    spark.read
    .option("header",              "true")
    .option("recursiveFileLookup", "true")
    .csv(glob_dir)
)
print("recursive    :", g3.count(), "rows")

## Write modes

Every `df.write` call accepts a mode that controls what happens when the destination already exists.

| Mode | Behavior |
|---|---|
| `overwrite` | Replace existing data completely |
| `append` | Add rows to existing data |
| `ignore` | Do nothing if destination exists (no error) |
| `error` (default) | Raise an error if destination exists |

Cells below use `overwrite` so they're idempotent — re-running the notebook does not pile up duplicates.

## Writing the five formats

Same DataFrame, written five ways. We pick a small slice (the first 100 customers) so the file outputs stay manageable.

In [ ]:
# A small slice we'll write five different ways
sample = customers.limit(100)

sample.write.mode("overwrite").option("header", "true").csv(f"{OUT_DIR}/customers_csv")
sample.write.mode("overwrite").json(f"{OUT_DIR}/customers_json")
sample.write.mode("overwrite").parquet(f"{OUT_DIR}/customers_parquet")
sample.write.mode("overwrite").orc(f"{OUT_DIR}/customers_orc")
sample.write.mode("overwrite").format("delta").save(f"{OUT_DIR}/customers_delta")

for fmt in ["csv", "json", "parquet", "orc", "delta"]:
    folder = f"{OUT_DIR}/customers_{fmt}"
    files  = [f for f in os.listdir(folder) if not f.startswith("_") and not f.startswith(".")]
    print(f"{fmt:8s} -> {len(files):2d} file(s)")

## `partitionBy` — the partition-pruning win

`partitionBy("col")` writes one subdirectory per distinct value of `col`. A query that filters on that column reads only the relevant subdirectories — Spark never opens the rest. This is **partition pruning**, and it's the single biggest write-time decision for downstream read performance.

```
data/_write_examples/transactions_by_status/
    status=APPROVED/  part-00000.parquet
    status=DECLINED/  part-00000.parquet
    status=REVERSED/  part-00000.parquet
```

Pick a column with **low cardinality** — a few dozen distinct values at most. Partitioning by `customer_id` would create millions of tiny directories.

In [ ]:
# Partition card transactions by status. Low cardinality = good candidate.
out = f"{OUT_DIR}/transactions_by_status"
(
    card_transactions
    .write.mode("overwrite")
    .partitionBy("status")
    .parquet(out)
)

# Inspect the directory tree — one subdir per status value.
for d in sorted(os.listdir(out)):
    if d.startswith("_"):
        continue
    print(d)

# Read back with a status filter — Spark skips other partitions automatically.
approved = spark.read.parquet(out).filter("status = 'APPROVED'")
print("approved rows:", approved.count())

## `bucketBy` — bucketing for join-friendly layouts

`bucketBy(n, "col")` hashes rows into `n` files per partition, fixing the column's value-to-file mapping. Two reasons to use it:

- Subsequent **joins** on that column can avoid a shuffle if both sides are bucketed identically (same `n`, same key).
- **Aggregations** on the bucket column can run per-bucket without a network step.

Catch: `bucketBy` only works with `saveAsTable` — it needs the metastore to record the bucketing metadata. Plain `save(path)` ignores it.

In [ ]:
# bucketBy needs saveAsTable — register a small bucketed table.
spark.sql("DROP TABLE IF EXISTS card_transactions_bucketed")

(
    card_transactions
    .write.mode("overwrite")
    .bucketBy(8, "card_id")
    .sortBy("card_id")
    .saveAsTable("card_transactions_bucketed")
)

# What the catalog knows about the table.
spark.sql("DESCRIBE FORMATTED card_transactions_bucketed").show(40, truncate=False)
print("rows:", spark.table("card_transactions_bucketed").count())

spark.sql("DROP TABLE card_transactions_bucketed")

## `saveAsTable` vs `save` — catalog vs filesystem

- **`save(path)`** writes files. Result: a directory of files. The catalog has no record of them; you reload with `spark.read.format(...).load(path)`.
- **`saveAsTable(name)`** writes files *and* registers the table in `spark.catalog`. You can then `spark.table(name)` or query it in SQL: `SELECT * FROM name`.

Use `saveAsTable` for SQL-style discoverability (anything anyone else might query). Use `save` for file-only outputs — intermediate staging, exports for downstream tools that don't read the catalog.

In [ ]:
# Two writes of the same DataFrame.
spark.sql("DROP TABLE IF EXISTS sample_catalog")
sample.write.mode("overwrite").parquet(f"{OUT_DIR}/sample_filesystem")  # files only
sample.write.mode("overwrite").saveAsTable("sample_catalog")            # files + catalog entry

# The catalog only knows about the saveAsTable target.
tables = [t.name for t in spark.catalog.listTables() if t.name.startswith("sample")]
print("catalog tables :", tables)

# Both can be read; only the catalog one is queryable by name.
print("via path  :", spark.read.parquet(f"{OUT_DIR}/sample_filesystem").count())
print("via name  :", spark.table("sample_catalog").count())
print("via SQL   :", spark.sql("SELECT COUNT(*) FROM sample_catalog").collect()[0][0])

spark.sql("DROP TABLE sample_catalog")

## `coalesce` vs `repartition` before writing

Both control the number of output files. The choice affects write speed, output file sizes, and downstream read parallelism.

| | `coalesce(n)` | `repartition(n)` |
|---|---|---|
| Shuffle | No (narrow transformation) | Yes (full shuffle) |
| Speed | Fast | Slow |
| Output sizes | Possibly uneven | Roughly equal |
| When to use | Reducing partition count, no rebalancing needed | Increasing partitions or rebalancing skewed data |

The classic case: `df.coalesce(1).write...` to produce a single output file. Cheap, but if the upstream stage had wildly uneven partitions, the one output partition holds all of it.

In [ ]:
# coalesce(1) — a single output file, no shuffle
sample.coalesce(1).write.mode("overwrite").parquet(f"{OUT_DIR}/sample_coalesce_1")

# repartition(4) — four roughly equal files, full shuffle
sample.repartition(4).write.mode("overwrite").parquet(f"{OUT_DIR}/sample_repartition_4")

def count_data_files(folder):
    return len([f for f in os.listdir(folder) if f.startswith("part-")])

print("coalesce(1)    files:", count_data_files(f"{OUT_DIR}/sample_coalesce_1"))
print("repartition(4) files:", count_data_files(f"{OUT_DIR}/sample_repartition_4"))

## JDBC briefly

For relational sources (Postgres, MySQL, SQL Server, Oracle) Spark uses the JDBC connector. The pattern:

```python
df = (
    spark.read.format("jdbc")
    .option("url",      "jdbc:postgresql://host:5432/db")
    .option("dbtable",  "schema.table_name")        # or .option("query", "SELECT ...")
    .option("user",     "...")
    .option("password", "...")
    .option("driver",   "org.postgresql.Driver")
    .load()
)
```

For **parallel reads**, set four extra options that tell Spark how to slice the query into N concurrent connections:

| Option | What it does |
|---|---|
| `partitionColumn` | Numeric/date column to slice on (must be indexed for sanity) |
| `lowerBound`, `upperBound` | Range of `partitionColumn` to cover |
| `numPartitions` | How many concurrent JDBC connections to open |

Without these, the read is single-threaded — one connection pulls everything to one executor. Always set them on tables larger than a few hundred MB.

Writing is symmetric: `df.write.format("jdbc").option(...).mode("append").save()`.

Cloud paths (`s3a://bucket/...`, `abfss://...`, `gs://...`) follow the exact same `spark.read.format(...).load(url)` pattern — only the URL scheme differs. The format options stay identical.

In [ ]:
# Optional cleanup — uncomment to remove every output written by this notebook.
# shutil.rmtree(OUT_DIR,   ignore_errors=True)
# shutil.rmtree(WAREHOUSE, ignore_errors=True)
spark.stop()